In [0]:
# =============================================================================
# NOTEBOOK  : 02_create_tables
# PURPOSE   : Create all Bronze and Silver Delta tables
# PROJECT   : azure3_team7_project — Retail Supply Chain & Inventory
# RUN       : Once, after 00_setup_catalog_schemas_volumes
#             and 01_create_audit_table
# NOTE      : Schema driven from schema_registry.py — single source of truth
#             Gold tables added later after silver is complete
#             CHECK constraints use spark.sql — Delta Python API does not
#             support constraints yet
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json

_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"

sys.path.insert(0, PROJECT_ROOT)
 
from utils.schema_registry import (
    # Bronze
    BRONZE_CUSTOMERS_SCHEMA,
    BRONZE_PRODUCT_MASTER_SCHEMA,
    BRONZE_STORE_MASTER_SCHEMA,
    BRONZE_SUPPLIER_MASTER_SCHEMA,
    BRONZE_PURCHASE_ORDERS_SCHEMA,
    BRONZE_STORE_INVENTORY_SCHEMA,
    BRONZE_POS_TRANSACTIONS_SCHEMA,
    BRONZE_WAREHOUSE_INVENTORY_SCHEMA,
    # Silver
    SILVER_CUSTOMERS_SCHEMA,
    SILVER_PRODUCT_MASTER_SCHEMA,
    SILVER_STORE_MASTER_SCHEMA,
    SILVER_SUPPLIER_MASTER_SCHEMA,
    SILVER_PURCHASE_ORDERS_SCHEMA,
    SILVER_STORE_INVENTORY_SCHEMA,
    SILVER_POS_TRANSACTIONS_SCHEMA,
    SILVER_WAREHOUSE_INVENTORY_SCHEMA,
    # Gold
    # Dimensions
    GOLD_DIM_PRODUCT_SCHEMA,
    GOLD_DIM_STORE_SCHEMA,
    GOLD_DIM_SUPPLIER_SCHEMA,
    GOLD_DIM_CUSTOMER_SCHEMA,
    # Facts
    GOLD_FACT_INVENTORY_FULL_SCHEMA,
    GOLD_FACT_DAILY_INVENTORY_SCHEMA,
    GOLD_FACT_INVENTORY_KPIS_SCHEMA,
    GOLD_FACT_DEMAND_TRENDS_SCHEMA,
    GOLD_FACT_CUSTOMER_SALES_SCHEMA,
    GOLD_FACT_SUPPLIER_PERFORMANCE_SCHEMA,
)
 
from delta.tables import DeltaTable
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
# Standard table properties applied to every table
TBLPROPERTIES = {
    "delta.autoOptimize.optimizeWrite": "true",
    "delta.autoOptimize.autoCompact":   "true",
    "delta.enableChangeDataFeed":       "true",
}
 
def create_table(name, schema, comment, partition_cols=None, properties=None):
    """
    Creates a Delta table if it does not already exist.
    Reads schema from schema_registry — no column definitions here.
    """
    props = properties or TBLPROPERTIES
    builder = (
        DeltaTable.createIfNotExists(spark)
        .tableName(name)
        .addColumns(schema)
        .comment(comment)
    )
    for k, v in props.items():
        builder = builder.property(k, v)
    if partition_cols:
        builder = builder.partitionedBy(*partition_cols)
    builder.execute()
    print(f"[CREATED] {name}")
 
 
def add_constraint(table, constraint_name, condition):
    """
    Adds a CHECK constraint to a table.
    Uses spark.sql — Delta Python API does not support constraints.
    Wrapped in try/except — constraint may already exist on reruns.
    """
    try:
        spark.sql(f"""
            ALTER TABLE {table}
            ADD CONSTRAINT {constraint_name}
            CHECK ({condition})
        """)
        print(f"  [CONSTRAINT] {constraint_name} added to {table}")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"  [CONSTRAINT] {constraint_name} already exists — skipping")
        else:
            raise

In [0]:
# ── Creating Bronze Tables ──────────────────────────────────────────────────

print("=" * 60)
print("CREATING BRONZE TABLES")
print("=" * 60)
 
# bronze.customers
# Partitioned by ingested_date
# preferred_categories kept as raw JSON string — parsed in silver
create_table(
    name          = "azure3_team7_project.bronze.customers",
    schema        = BRONZE_CUSTOMERS_SCHEMA,
    comment       = "Bronze: Customer data (anonymized) from CSV. Append-only.",
    partition_cols = ["ingested_date"]
)
 
# bronze.product_master
# Not partitioned — small table (50k rows), weekly incremental
# expiry_date stored as INT (days since epoch) — cast in silver
# dimensions stored as raw string "LxWxH" — parsed in silver
create_table(
    name    = "azure3_team7_project.bronze.product_master",
    schema  = BRONZE_PRODUCT_MASTER_SCHEMA,
    comment = "Bronze: Product master from Parquet. Append-only. SCD2 handled in silver."
)
 
# bronze.store_master
# Not partitioned — tiny table (500 rows), weekly incremental
create_table(
    name    = "azure3_team7_project.bronze.store_master",
    schema  = BRONZE_STORE_MASTER_SCHEMA,
    comment = "Bronze: Store master reference data from CSV. SCD2 handled in silver."
)
 
# bronze.supplier_master
# Not partitioned — tiny table (200 rows), weekly incremental
create_table(
    name    = "azure3_team7_project.bronze.supplier_master",
    schema  = BRONZE_SUPPLIER_MASTER_SCHEMA,
    comment = "Bronze: Supplier master reference data from CSV. SCD2 handled in silver."
)
 
# bronze.purchase_orders
# Partitioned by ingested_date — daily + CDC source
# delivery_notes kept as raw nested JSON string — flattened in silver
create_table(
    name           = "azure3_team7_project.bronze.purchase_orders",
    schema         = BRONZE_PURCHASE_ORDERS_SCHEMA,
    comment        = "Bronze: Purchase orders from CSV with nested JSON delivery_notes. Append-only.",
    partition_cols = ["ingested_date"]
)
 
# bronze.store_inventory
# Partitioned by ingested_date — every 15 min snapshots
# temperature_reading kept as raw nested JSON string — flattened in silver
create_table(
    name           = "azure3_team7_project.bronze.store_inventory",
    schema         = BRONZE_STORE_INVENTORY_SCHEMA,
    comment        = "Bronze: Store inventory snapshots from JSONL every 15 mins. Append-only.",
    partition_cols = ["ingested_date"]
)
 
# bronze.pos_transactions
# Partitioned by ingested_date — real-time stream + daily batch
# _source column distinguishes stream vs batch rows
# Both pos_realtime_stream and pos_batch_csv land in same table
create_table(
    name           = "azure3_team7_project.bronze.pos_transactions",
    schema         = BRONZE_POS_TRANSACTIONS_SCHEMA,
    comment        = "Bronze: POS transactions from JSONL stream and CSV batch. Append-only.",
    partition_cols = ["ingested_date"]
)
 
# Constraints on bronze.pos_transactions
# Only table with constraints in bronze — _source is critical for pipeline routing
add_constraint(
    "azure3_team7_project.bronze.pos_transactions",
    "valid_source",
    "_source IN ('pos_realtime_stream', 'pos_batch_csv')"
)
 
# bronze.warehouse_inventory
# Partitioned by ingested_date — hourly snapshots
# last_updated stored as STRING (Parquet BINARY) — cast in silver
# stock columns as BIGINT — matches actual Parquet source types
create_table(
    name           = "azure3_team7_project.bronze.warehouse_inventory",
    schema         = BRONZE_WAREHOUSE_INVENTORY_SCHEMA,
    comment        = "Bronze: Warehouse inventory snapshots from Parquet. Append-only.",
    partition_cols = ["ingested_date"]
)
 
print("\n[DONE] All bronze tables created")


In [0]:
# ── Creating Silver Tables ─────────────────────────────────────────────────


print("=" * 60)
print("CREATING SILVER TABLES")
print("=" * 60)
 
# silver.customers
# Merge key: customer_id
# preferred_categories parsed from JSON string to ARRAY<STRING> in pipeline
# first_purchase_date immutable — set once, never updated
create_table(
    name    = "azure3_team7_project.silver.customers",
    schema  = SILVER_CUSTOMERS_SCHEMA,
    comment = "Silver: Cleaned customer data. preferred_categories parsed to array. Merged on customer_id."
)
 
add_constraint(
    "azure3_team7_project.silver.customers",
    "valid_loyalty_tier",
    "loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')"
)
 
# silver.product_master — SCD Type 2
# Merge key: product_id + scd_hash
# scd_hash detects changes — new hash = new version
# is_current=True = latest active version
# dimensions parsed into length, width, height in pipeline
create_table(
    name    = "azure3_team7_project.silver.product_master",
    schema  = SILVER_PRODUCT_MASTER_SCHEMA,
    comment = "Silver: Product master with SCD Type 2 history. is_current=true for latest version."
)
 
add_constraint(
    "azure3_team7_project.silver.product_master",
    "valid_product_status",
    "status IN ('active', 'discontinued')"
)
 
# silver.store_master — SCD Type 2
# Merge key: store_id + scd_hash
create_table(
    name    = "azure3_team7_project.silver.store_master",
    schema  = SILVER_STORE_MASTER_SCHEMA,
    comment = "Silver: Store master with SCD Type 2 history. is_current=true for latest version."
)
 
add_constraint(
    "azure3_team7_project.silver.store_master",
    "valid_store_type",
    "store_type IN ('Warehouse', 'Superstore', 'Express', 'Online')"
)
 
# silver.supplier_master — SCD Type 2
# Merge key: supplier_id + scd_hash
create_table(
    name    = "azure3_team7_project.silver.supplier_master",
    schema  = SILVER_SUPPLIER_MASTER_SCHEMA,
    comment = "Silver: Supplier master with SCD Type 2 history. is_current=true for latest version."
)
 
add_constraint(
    "azure3_team7_project.silver.supplier_master",
    "valid_performance_rating",
    "performance_rating BETWEEN 0 AND 5"
)
 
add_constraint(
    "azure3_team7_project.silver.supplier_master",
    "valid_delivery_pct",
    "on_time_delivery_pct BETWEEN 0 AND 100"
)
 
# silver.purchase_orders
# Merge key: po_id (CDC — same PO arrives with status updates)
# Partitioned by order_year_month — derived from order_date in pipeline
# delivery_notes flattened into 4 columns: carrier, tracking_number,
# delivery_status, delivery_notes_text
# Updatable: status only — all other columns immutable once PO created
create_table(
    name           = "azure3_team7_project.silver.purchase_orders",
    schema         = SILVER_PURCHASE_ORDERS_SCHEMA,
    comment        = "Silver: Purchase orders with flattened delivery_notes. Merged on po_id for CDC.",
    partition_cols = ["order_year_month"]
)
 
add_constraint(
    "azure3_team7_project.silver.purchase_orders",
    "valid_po_status",
    "status IN ('pending', 'shipped', 'delivered', 'cancelled')"
)
 
add_constraint(
    "azure3_team7_project.silver.purchase_orders",
    "positive_quantity_ordered",
    "quantity_ordered > 0"
)
 
# silver.store_inventory
# Merge key: store_id + product_id (latest snapshot per pair)
# temperature_reading flattened into 3 columns: sensor_id,
# temperature_celsius, humidity
# All columns updatable — pure snapshot table
create_table(
    name    = "azure3_team7_project.silver.store_inventory",
    schema  = SILVER_STORE_INVENTORY_SCHEMA,
    comment = "Silver: Latest store inventory per store+product. temperature_reading flattened."
)
 
add_constraint(
    "azure3_team7_project.silver.store_inventory",
    "non_negative_quantity",
    "current_quantity >= 0"
)
 
# silver.pos_transactions
# Merge key: transaction_id
# Partitioned by transaction_date — derived from transaction_ts in pipeline
# Stream pipeline: INSERT only, no updates, no deletes
# Batch pipeline: INSERT new, UPDATE corrections, DELETE cancelled
# timestamp renamed to transaction_ts — avoids Spark reserved word
create_table(
    name           = "azure3_team7_project.silver.pos_transactions",
    schema         = SILVER_POS_TRANSACTIONS_SCHEMA,
    comment        = "Silver: Cleaned, deduplicated POS transactions. Merged on transaction_id.",
    partition_cols = ["transaction_date"]
)
 
add_constraint(
    "azure3_team7_project.silver.pos_transactions",
    "valid_channel",
    "channel IN ('online', 'offline')"
)
 
add_constraint(
    "azure3_team7_project.silver.pos_transactions",
    "positive_amount",
    "total_amount >= 0"
)
 
add_constraint(
    "azure3_team7_project.silver.pos_transactions",
    "positive_quantity",
    "quantity > 0"
)
 
# silver.warehouse_inventory
# Merge key: warehouse_id + product_id (latest snapshot per pair)
# available_stock recomputed as current_stock - reserved_stock in pipeline
# source value not trusted
# All stock columns updatable — pure snapshot table
create_table(
    name    = "azure3_team7_project.silver.warehouse_inventory",
    schema  = SILVER_WAREHOUSE_INVENTORY_SCHEMA,
    comment = "Silver: Latest warehouse inventory per warehouse+product. Merged on warehouse_id+product_id."
)
 
add_constraint(
    "azure3_team7_project.silver.warehouse_inventory",
    "non_negative_current_stock",
    "current_stock >= 0"
)
 
add_constraint(
    "azure3_team7_project.silver.warehouse_inventory",
    "non_negative_reserved_stock",
    "reserved_stock >= 0"
)
 
print("\n[DONE] All silver tables created")

In [0]:
# ── Creating Gold Tables ─────────────────────────────────────────────────

# ── Dimension Tables ───────────────────────────────────────────────────────
print("=" * 60)
print("CREATING GOLD DIMENSION TABLES")
print("=" * 60)

# gold.dim_product
# Weekly refresh — full overwrite of current product state
# Not partitioned — 50K rows, too small to benefit from partitioning
create_table(
    name    = cfg["tables"]["gold_dim_product"],
    schema  = GOLD_DIM_PRODUCT_SCHEMA,
    comment = "Gold: Product dimension. Current active products with profit margin and price tier. Weekly refresh."
)

add_constraint(
    cfg["tables"]["gold_dim_product"],
    "gold_valid_product_status",
    "status IN ('active', 'discontinued')"
)

add_constraint(
    cfg["tables"]["gold_dim_product"],
    "gold_valid_price_tier",
    "price_tier IN ('Budget', 'Mid-range', 'Premium')"
)

# gold.dim_store
# Weekly refresh — full overwrite of current store state
# Not partitioned — 500 rows
create_table(
    name    = cfg["tables"]["gold_dim_store"],
    schema  = GOLD_DIM_STORE_SCHEMA,
    comment = "Gold: Store dimension. Current stores with size tier and years in operation. Weekly refresh."
)

add_constraint(
    cfg["tables"]["gold_dim_store"],
    "gold_valid_store_size_tier",
    "store_size_tier IN ('Large', 'Medium', 'Small')"
)

# gold.dim_supplier
# Weekly refresh — full overwrite of current supplier state
# Not partitioned — 200 rows
create_table(
    name    = cfg["tables"]["gold_dim_supplier"],
    schema  = GOLD_DIM_SUPPLIER_SCHEMA,
    comment = "Gold: Supplier dimension. Current suppliers with baseline risk tier and performance band. Weekly refresh."
)

add_constraint(
    cfg["tables"]["gold_dim_supplier"],
    "gold_valid_baseline_risk_tier",
    "baseline_risk_tier IN ('Low', 'Medium', 'High')"
)

add_constraint(
    cfg["tables"]["gold_dim_supplier"],
    "gold_valid_performance_band",
    "performance_band IN ('Excellent', 'Good', 'Average')"
)

# gold.dim_customer
# Daily refresh — MERGE on customer_id (incremental)
# Not partitioned — 100K rows, queried by customer_id not date
create_table(
    name    = cfg["tables"]["gold_dim_customer"],
    schema  = GOLD_DIM_CUSTOMER_SCHEMA,
    comment = "Gold: Customer dimension. One row per customer with attributes. Daily MERGE refresh."
)

add_constraint(
    cfg["tables"]["gold_dim_customer"],
    "gold_valid_loyalty_tier",
    "loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')"
)

print("\n[DONE] All gold dimension tables created")


# ── Fact Tables ───────────────────────────────────────────────────────
print("=" * 60)
print("CREATING GOLD FACT TABLES")
print("=" * 60)
 
# gold.fact_inventory_full
# Intermediate gold table — joins POS + warehouse + product master
# Feeds: fact_inventory_kpis and fact_demand_trends
# Grain: store_id + product_id + snapshot_date
create_table(
    name           = cfg["tables"]["gold_fact_inventory_full"],
    schema         = GOLD_FACT_INVENTORY_FULL_SCHEMA,
    comment        = (
        "Gold: POS sales joined with warehouse stock and product master. "
        "Grain: store_id + product_id + snapshot_date. "
        "Daily refresh. Feeds fact_inventory_kpis and fact_demand_trends."
    ),
    partition_cols = ["snapshot_date"]
)
add_constraint(cfg["tables"]["gold_fact_inventory_full"], "gold_inv_full_valid_status",
    "status IN ('active', 'discontinued')")

# gold.fact_daily_inventory
# Daily inventory fact with store shelf stock
# Grain: store_id + product_id + transaction_date
create_table(
    name           = cfg["tables"]["gold_fact_daily_inventory"],
    schema         = GOLD_FACT_DAILY_INVENTORY_SCHEMA,
    comment        = (
        "Gold: Daily inventory fact with store shelf stock. "
        "Grain: store_id + product_id + transaction_date. "
        "Daily refresh. Partitioned by transaction_date."
    ),
    partition_cols = ["transaction_date"]
)
 
# gold.fact_inventory_kpis
# Built on top of fact_inventory_full
# Adds rolling window KPIs: 30d turnover, overstock risk, dead stock, health
# Grain: store_id + product_id + snapshot_date
create_table(
    name           = cfg["tables"]["gold_fact_inventory_kpis"],
    schema         = GOLD_FACT_INVENTORY_KPIS_SCHEMA,
    comment        = (
        "Gold: Rolling inventory KPIs (30d, 90d windows). "
        "Built from fact_inventory_full. "
        "Grain: store_id + product_id + snapshot_date. Daily refresh."
    ),
    partition_cols = ["snapshot_date"]
)
add_constraint(cfg["tables"]["gold_fact_inventory_kpis"], "gold_valid_inventory_health",
    "inventory_health IN ('Healthy', 'Warning — Low Stock', 'Warning — Overstock', 'Critical — Dead Stock', 'Critical — Stockout')")
 
# gold.fact_demand_trends
# Built on top of fact_inventory_full
# Rolling demand averages (7d, 30d, 90d), seasonality index, trend direction
# Grain: store_id + product_id + snapshot_date
create_table(
    name           = cfg["tables"]["gold_fact_demand_trends"],
    schema         = GOLD_FACT_DEMAND_TRENDS_SCHEMA,
    comment        = (
        "Gold: Demand analytics with rolling averages and trend signals. "
        "Built from fact_inventory_full. "
        "Grain: store_id + product_id + snapshot_date. "
        "Daily refresh. Partitioned by year_month."
    ),
    partition_cols = ["year_month"]
)
add_constraint(cfg["tables"]["gold_fact_demand_trends"], "gold_valid_trend_direction",
    "trend_direction IN ('Rising', 'Falling', 'Stable')")
 
# gold.fact_customer_sales
# Enriched POS transactions joined with customer attributes
# Grain: one row per transaction_id
create_table(
    name           = cfg["tables"]["gold_fact_customer_sales"],
    schema         = GOLD_FACT_CUSTOMER_SALES_SCHEMA,
    comment        = (
        "Gold: Enriched POS transactions with customer attributes. "
        "Grain: transaction_id. Daily refresh. Partitioned by transaction_date."
    ),
    partition_cols = ["transaction_date"]
)
add_constraint(cfg["tables"]["gold_fact_customer_sales"], "gold_cs_valid_channel",
    "channel IN ('online', 'offline')")
add_constraint(cfg["tables"]["gold_fact_customer_sales"], "gold_cs_positive_amount",
    "total_amount >= 0")
add_constraint(cfg["tables"]["gold_fact_customer_sales"], "gold_cs_positive_quantity",
    "quantity > 0")
 
# gold.fact_supplier_performance
# Supplier KPIs aggregated across all PO history
# Grain: one row per supplier_id
create_table(
    name    = cfg["tables"]["gold_fact_supplier_performance"],
    schema  = GOLD_FACT_SUPPLIER_PERFORMANCE_SCHEMA,
    comment = (
        "Gold: Supplier performance KPIs across all purchase order history. "
        "Grain: supplier_id. Daily full overwrite."
    )
)
add_constraint(cfg["tables"]["gold_fact_supplier_performance"], "gold_valid_risk_tier",
    "actual_risk_tier IN ('Low', 'Medium', 'High')")
 
print("\n[DONE] All gold fact tables created")


In [0]:
# ── Verify all tables ─────────────────────────────────────────────────


print("=" * 60)
print("VERIFICATION")
print("=" * 60)
 
print("\nBronze tables:")
spark.sql("SHOW TABLES IN azure3_team7_project.bronze").show(truncate=False)
 
print("\nSilver tables:")
spark.sql("SHOW TABLES IN azure3_team7_project.silver").show(truncate=False)

print("\nGold tables:")
spark.sql("SHOW TABLES IN azure3_team7_project.gold").show(truncate=False)

print("\nPlatform tables:")
spark.sql("SHOW TABLES IN azure3_team7_project.platform").show(truncate=False)